In [ ]:
import pandas as pd

# Load both files - cc_num forced to string to prevent scientific notation corruption
train = pd.read_csv('fraudTrain.csv', dtype={'cc_num': str})
test = pd.read_csv('fraudTest.csv', dtype={'cc_num': str})

# Quick check before merging - confirm same columns
print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Same columns?", list(train.columns) == list(test.columns))

Train shape: (1296675, 23)
Test shape: (555719, 23)
Same columns? True


In [ ]:
df = pd.concat([train, test], ignore_index=True)
print("Merged shape:", df.shape)
df.head()

Merged shape: (1852394, 23)


,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,...,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0
1,1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,...,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0
2,2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,...,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0
3,3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,...,46.2306,-112.1138,1939,Patent attorney,1967-01-12,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0
4,4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,...,38.4207,-79.4629,99,Dance movement psychotherapist,1986-03-28,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0


In [ ]:
df = df.drop(columns=['Unnamed: 0'], errors='ignore')
print(df.shape)  # should be (1604294, 22) after dropping

(1852394, 22)


In [ ]:
df.info()                            # dtypes + non-null counts for all columns
df.isnull().sum()                    # nulls per column
df['trans_num'].duplicated().sum()   # should be 0 if trans_num is a true unique ID
df['is_fraud'].value_counts(normalize=True) * 100   # fraud vs legit %

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1852394 entries, 0 to 1852393
Data columns (total 22 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   trans_date_trans_time  object 
 1   cc_num                 object 
 2   merchant               object 
 3   category               object 
 4   amt                    float64
 5   first                  object 
 6   last                   object 
 7   gender                 object 
 8   street                 object 
 9   city                   object 
 10  state                  object 
 11  zip                    int64  
 12  lat                    float64
 13  long                   float64
 14  city_pop               int64  
 15  job                    object 
 16  dob                    object 
 17  trans_num              object 
 18  unix_time              int64  
 19  merch_lat              float64
 20  merch_long             float64
 21  is_fraud               int64  
dtypes: float64(5), int

,proportion
is_fraud,
0,99.478999
1,0.521001


In [ ]:
print(df['cc_num'].dtype)
print(df['cc_num'].head())

object
0    2703186189652095
1        630423337322
2      38859492057661
3    3534093764340240
4     375534208663984
Name: cc_num, dtype: object


In [ ]:
# Convert to proper datetime
df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
df['dob'] = pd.to_datetime(df['dob'])

# Feature engineering
df['hour'] = df['trans_date_trans_time'].dt.hour
df['day_of_week'] = df['trans_date_trans_time'].dt.day_name()
df['month'] = df['trans_date_trans_time'].dt.month
df['year'] = df['trans_date_trans_time'].dt.year

# Age at time of transaction
df['age'] = (df['trans_date_trans_time'] - df['dob']).dt.days // 365

# Age bands
bins = [0, 25, 35, 45, 55, 65, 100]
labels = ['18-25', '26-35', '36-45', '46-55', '56-65', '65+']
df['age_band'] = pd.cut(df['age'], bins=bins, labels=labels)

# Sanity check
df[['trans_date_trans_time', 'dob', 'hour', 'day_of_week', 'month', 'age', 'age_band']].head()

,trans_date_trans_time,dob,hour,day_of_week,month,age,age_band
0,2019-01-01 00:00:18,1988-03-09,0,Tuesday,1,30,26-35
1,2019-01-01 00:00:44,1978-06-21,0,Tuesday,1,40,36-45
2,2019-01-01 00:00:51,1962-01-19,0,Tuesday,1,56,56-65
3,2019-01-01 00:01:16,1967-01-12,0,Tuesday,1,52,46-55
4,2019-01-01 00:03:06,1986-03-28,0,Tuesday,1,32,26-35


In [ ]:
print("Min age:", df['age'].min())
print("Max age:", df['age'].max())
print("Age band nulls:", df['age_band'].isnull().sum())
print("Hour range:", df['hour'].min(), "-", df['hour'].max())
print("Unique days:", df['day_of_week'].unique())

Min age: 13
Max age: 96
Age band nulls: 0
Hour range: 0 - 23
Unique days: ['Tuesday' 'Wednesday' 'Thursday' 'Friday' 'Saturday' 'Sunday' 'Monday']


In [ ]:
df.to_csv('fraud_cleaned.csv', index=False)
print("Exported. Final shape:", df.shape)  # expect (1604294, 28)

Exported. Final shape: (1852394, 28)


In [ ]:
# Verify the exported file directly, safely, without opening it in Excel
check = pd.read_csv('fraud_cleaned.csv', dtype={'cc_num': str}, nrows=5)
print(check['cc_num'])
print("Full row count check:")
with open('fraud_cleaned.csv', 'r') as f:
    row_count = sum(1 for line in f)
print("Total lines (including header):", row_count)  # expect 1604295 (1604294 rows + 1 header)

0    2703186189652095
1        630423337322
2      38859492057661
3    3534093764340240
4     375534208663984
Name: cc_num, dtype: object
Full row count check:
Total lines (including header): 1852395
